# Lab 02 - Muc tieu 2: Giao duc

**Thanh vien phu trach:** Hoang Quoc Viet (23120189)

Notebook nay thuc hien hai cong viec:

- **2.1:** Khao sat indicators giao duc trong WDI va xac nhan ma chi so kha dung.
- **2.2:** Tien xu ly du lieu giao duc: loc indicators, loai aggregate rows, xu ly missing values theo quy uoc nhom, chuan hoa cot va xuat file `wdi_education.csv`.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)


In [ ]:
cwd = Path.cwd()
if (cwd / "WDIEXCEL.xlsx").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "WDIEXCEL.xlsx").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Khong tim thay WDIEXCEL.xlsx o thu muc hien tai hoac thu muc cha.")

SOURCE_FILE = PROJECT_ROOT / "WDIEXCEL.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "data_output"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source file: {SOURCE_FILE.name}")
print(f"Output dir:  {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")


Project root: D:\Lab2_TQH
Source file: WDIEXCEL.xlsx
Output dir:  data_output


## 2.1. Khao sat indicators giao duc

Bon indicator duoc chon deu thuoc WDI va phu hop voi muc tieu phan tich ve tiep can giao duc, ket qua giao duc va dau tu cho giao duc:

| Indicator Code | Cot sau tien xu ly | Y nghia trong phan tich |
|---|---|---|
| `SE.PRM.NENR` | `Primary_Net_Enrollment` | Ty le nhap hoc dung do tuoi o bac tieu hoc |
| `SE.SEC.NENR` | `Secondary_Net_Enrollment` | Ty le nhap hoc dung do tuoi o bac trung hoc |
| `SE.ADT.LITR.ZS` | `Adult_Literacy_Rate` | Ty le biet chu cua nguoi tu 15 tuoi tro len |
| `SE.XPD.TOTL.GD.ZS` | `Education_Expenditure_GDP` | Chi tieu giao duc cua chinh phu theo % GDP |


In [ ]:
INDICATORS = {
    "SE.PRM.NENR": {
        "clean_name": "Primary_Net_Enrollment",
        "analysis_role": "Access to primary education",
    },
    "SE.SEC.NENR": {
        "clean_name": "Secondary_Net_Enrollment",
        "analysis_role": "Access to secondary education",
    },
    "SE.ADT.LITR.ZS": {
        "clean_name": "Adult_Literacy_Rate",
        "analysis_role": "Education outcome",
    },
    "SE.XPD.TOTL.GD.ZS": {
        "clean_name": "Education_Expenditure_GDP",
        "analysis_role": "Government education investment",
    },
}

indicator_codes = list(INDICATORS)
years = [str(year) for year in range(2000, 2024)]

series_cols = [
    "Series Code",
    "Topic",
    "Indicator Name",
    "Long definition",
    "Unit of measure",
    "Periodicity",
    "Source",
]
series = pd.read_excel(SOURCE_FILE, sheet_name="Series", usecols=series_cols)
indicator_audit = (
    series[series["Series Code"].isin(indicator_codes)]
    .copy()
    .assign(
        Clean_Column=lambda df: df["Series Code"].map(
            {code: meta["clean_name"] for code, meta in INDICATORS.items()}
        ),
        Analysis_Role=lambda df: df["Series Code"].map(
            {code: meta["analysis_role"] for code, meta in INDICATORS.items()}
        ),
        Available=True,
    )
    .sort_values("Series Code")
)

missing_codes = sorted(set(indicator_codes) - set(indicator_audit["Series Code"]))
if missing_codes:
    raise ValueError(f"Cac indicator khong co trong sheet Series: {missing_codes}")

indicator_audit.to_csv(OUTPUT_DIR / "wdi_education_indicator_audit.csv", index=False, encoding="utf-8-sig")
indicator_audit


,Series Code,Topic,Indicator Name,Long definition,Unit of measure,Periodicity,Source,Clean_Column,Analysis_Role,Available
802,SE.ADT.LITR.ZS,Education: Outcomes,"Literacy rate, adult total (% of people ages 15 and above)",Adult literacy rate is the percentage of people ages 15 and above who can both read and write with understanding a short simple statemen...,% of people ages 15 and above,Annual,"Data API, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://databrowser.uis.unesco.org/resources, note: The da...",Adult_Literacy_Rate,Education outcome,True
842,SE.PRM.NENR,Education: Participation,"School enrollment, primary (% net)",Net enrollment rate is the ratio of children of official school age who are enrolled in school to the population of the corresponding of...,NaN,Annual,"Stat Bulk Data Download Service, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://uis.unesco.org/bdds, publis...",Primary_Net_Enrollment,Access to primary education,True
901,SE.SEC.NENR,Education: Participation,"School enrollment, secondary (% net)",Net enrollment rate is the ratio of children of official school age who are enrolled in school to the population of the corresponding of...,NaN,Annual,"Stat Bulk Data Download Service, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://uis.unesco.org/bdds, publis...",Secondary_Net_Enrollment,Access to secondary education,True
951,SE.XPD.TOTL.GD.ZS,Education: Inputs,"Government expenditure on education, total (% of GDP)","General government expenditure on education (current, capital, and transfers) is expressed as a percentage of GDP. It includes expenditu...",% of GDP,Annual,"Data API, UN Educational, Scientific and Cultural Organization (UNESCO), uri: https://databrowser.uis.unesco.org/resources, note: The da...",Education_Expenditure_GDP,Government education investment,True


In [ ]:
country_cols = ["Country Code", "Short Name", "Table Name", "Region", "Income Group"]
country = pd.read_excel(SOURCE_FILE, sheet_name="Country", usecols=country_cols)

country_clean = country.rename(
    columns={
        "Country Code": "Country_Code",
        "Short Name": "Short_Name",
        "Table Name": "Country_Name_Table",
        "Income Group": "Income_Group",
    }
)

# Trong WDI, cac dong aggregate nhu World, income groups, regions thuong co Region bi trong.
real_countries = country_clean[country_clean["Region"].notna()].copy()
aggregate_rows = country_clean[country_clean["Region"].isna()].copy()

print(f"Country metadata rows: {len(country_clean):,}")
print(f"Real countries kept:    {len(real_countries):,}")
print(f"Aggregate rows removed: {len(aggregate_rows):,}")

region_summary = real_countries["Region"].value_counts().rename_axis("Region").reset_index(name="Country_Count")
region_summary


Country metadata rows: 265
Real countries kept:    217
Aggregate rows removed: 48


,Region,Country_Count
0,Europe & Central Asia,58
1,Sub-Saharan Africa,48
2,Latin America & Caribbean,42
3,East Asia & Pacific,37
4,Middle East & North Africa,23
5,South Asia,6
6,North America,3


In [ ]:
data_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"] + years
raw = pd.read_excel(SOURCE_FILE, sheet_name="Data", usecols=data_cols)

education_raw = raw[
    raw["Indicator Code"].isin(indicator_codes)
    & raw["Country Code"].isin(real_countries["Country_Code"])
].copy()

availability_rows = []
for code in indicator_codes:
    sub = education_raw[education_raw["Indicator Code"].eq(code)].copy()
    values = sub[years]
    yearly_non_null = values.notna().sum()
    availability_rows.append(
        {
            "Indicator_Code": code,
            "Clean_Column": INDICATORS[code]["clean_name"],
            "Indicator_Name": sub["Indicator Name"].iloc[0] if len(sub) else None,
            "Country_Rows": len(sub),
            "Countries_With_Any_Data": int(values.notna().any(axis=1).sum()),
            "Non_Null_Cells_2000_2023": int(values.notna().sum().sum()),
            "First_Year_With_Data": int(yearly_non_null[yearly_non_null.gt(0)].index.min()),
            "Last_Year_With_Data": int(yearly_non_null[yearly_non_null.gt(0)].index.max()),
        }
    )

availability = pd.DataFrame(availability_rows)
availability


,Indicator_Code,Clean_Column,Indicator_Name,Country_Rows,Countries_With_Any_Data,Non_Null_Cells_2000_2023,First_Year_With_Data,Last_Year_With_Data
0,SE.PRM.NENR,Primary_Net_Enrollment,"School enrollment, primary (% net)",217,188,2419,2000,2019
1,SE.SEC.NENR,Secondary_Net_Enrollment,"School enrollment, secondary (% net)",217,180,1891,2000,2019
2,SE.ADT.LITR.ZS,Adult_Literacy_Rate,"Literacy rate, adult total (% of people ages 15 and above)",217,155,892,2000,2023
3,SE.XPD.TOTL.GD.ZS,Education_Expenditure_GDP,"Government expenditure on education, total (% of GDP)",217,200,3325,2000,2023


## 2.2. Quy uoc tien xu ly

Cac buoc tien xu ly:

1. Chi giu 4 indicators giao duc da xac minh o muc 2.1.
2. Chi giu cac quoc gia that, loai aggregate rows bang metadata `Country.Region`.
3. Gioi han giai doan phan tich `2000-2023`.
4. Voi tung cap `Country_Code` va `Indicator_Code`, chi noi suy tuyen tinh cac khoang missing nam **ben trong chuoi** co do dai toi da 2 nam.
5. Neu mot cap country-indicator co khoang missing ben trong dai hon 2 nam thi loai cap do khoi du lieu sach de tranh tao du lieu gia.
6. Khong extrapolate cho cac nam dau/cuoi nam ngoai pham vi quan sat cua indicator.
7. Xuat du lieu dang wide de nap vao Tableau, moi dong la mot `Country_Code` - `Year`.


In [ ]:
def max_missing_run(values: pd.Series) -> int:
    max_run = 0
    current = 0
    for value in values:
        if pd.isna(value):
            current += 1
            max_run = max(max_run, current)
        else:
            current = 0
    return max_run


def max_internal_missing_run(values: pd.Series) -> int:
    observed_positions = np.flatnonzero(values.notna().to_numpy())
    if len(observed_positions) == 0:
        return -1
    if len(observed_positions) == 1:
        return 0
    internal = values.iloc[observed_positions[0] : observed_positions[-1] + 1]
    return max_missing_run(internal)


def clean_country_indicator(row: pd.Series) -> tuple[str, pd.Series, int]:
    values = pd.to_numeric(row[years], errors="coerce")
    observed_positions = np.flatnonzero(values.notna().to_numpy())

    if len(observed_positions) == 0:
        return "dropped_no_data", values, 0

    max_gap = max_internal_missing_run(values)
    if max_gap > 2:
        return "dropped_internal_gap_gt_2", values, 0

    cleaned = values.copy()
    first_pos = observed_positions[0]
    last_pos = observed_positions[-1]
    internal = cleaned.iloc[first_pos : last_pos + 1]
    interpolated_internal = internal.interpolate(method="linear", limit=2, limit_area="inside")
    cleaned.iloc[first_pos : last_pos + 1] = interpolated_internal

    imputed_count = int(values.isna().sum() - cleaned.isna().sum())
    return "kept", cleaned, imputed_count


In [ ]:
long_records = []
cleaning_records = []

for _, row in education_raw.iterrows():
    status, cleaned_values, imputed_count = clean_country_indicator(row)
    original_values = pd.to_numeric(row[years], errors="coerce")
    max_internal_gap = max_internal_missing_run(original_values)

    cleaning_records.append(
        {
            "Country_Code": row["Country Code"],
            "Country_Name": row["Country Name"],
            "Indicator_Code": row["Indicator Code"],
            "Indicator_Name": row["Indicator Name"],
            "Clean_Column": INDICATORS[row["Indicator Code"]]["clean_name"],
            "Status": status,
            "Observed_Cells_Before": int(original_values.notna().sum()),
            "Observed_Cells_After": int(cleaned_values.notna().sum()) if status == "kept" else 0,
            "Imputed_Cells": imputed_count if status == "kept" else 0,
            "Max_Internal_Missing_Run": max_internal_gap,
        }
    )

    if status != "kept":
        continue

    value_col = INDICATORS[row["Indicator Code"]]["clean_name"]
    for year, value in cleaned_values.items():
        if pd.notna(value):
            long_records.append(
                {
                    "Country_Name": row["Country Name"],
                    "Country_Code": row["Country Code"],
                    "Year": int(year),
                    "Indicator_Code": row["Indicator Code"],
                    "Indicator_Name": row["Indicator Name"],
                    "Metric": value_col,
                    "Value": float(value),
                }
            )

education_long = pd.DataFrame(long_records)
cleaning_report = pd.DataFrame(cleaning_records)

cleaning_report.to_csv(OUTPUT_DIR / "wdi_education_cleaning_report.csv", index=False, encoding="utf-8-sig")

status_summary = (
    cleaning_report
    .groupby(["Indicator_Code", "Clean_Column", "Status"], as_index=False)
    .agg(
        Country_Indicator_Count=("Country_Code", "count"),
        Observed_Cells_Before=("Observed_Cells_Before", "sum"),
        Observed_Cells_After=("Observed_Cells_After", "sum"),
        Imputed_Cells=("Imputed_Cells", "sum"),
    )
)
status_summary


,Indicator_Code,Clean_Column,Status,Country_Indicator_Count,Observed_Cells_Before,Observed_Cells_After,Imputed_Cells
0,SE.ADT.LITR.ZS,Adult_Literacy_Rate,dropped_internal_gap_gt_2,114,646,0,0
1,SE.ADT.LITR.ZS,Adult_Literacy_Rate,dropped_no_data,62,0,0,0
2,SE.ADT.LITR.ZS,Adult_Literacy_Rate,kept,41,246,294,48
3,SE.PRM.NENR,Primary_Net_Enrollment,dropped_internal_gap_gt_2,45,412,0,0
4,SE.PRM.NENR,Primary_Net_Enrollment,dropped_no_data,29,0,0,0
5,SE.PRM.NENR,Primary_Net_Enrollment,kept,143,2007,2144,137
6,SE.SEC.NENR,Secondary_Net_Enrollment,dropped_internal_gap_gt_2,47,372,0,0
7,SE.SEC.NENR,Secondary_Net_Enrollment,dropped_no_data,37,0,0,0
8,SE.SEC.NENR,Secondary_Net_Enrollment,kept,133,1519,1637,118
9,SE.XPD.TOTL.GD.ZS,Education_Expenditure_GDP,dropped_internal_gap_gt_2,61,847,0,0


In [ ]:
education_wide = (
    education_long
    .pivot_table(
        index=["Country_Name", "Country_Code", "Year"],
        columns="Metric",
        values="Value",
        aggfunc="first",
    )
    .reset_index()
)
education_wide.columns.name = None

education_wide = education_wide.merge(
    real_countries[["Country_Code", "Region", "Income_Group"]],
    on="Country_Code",
    how="left",
)

metric_columns = [INDICATORS[code]["clean_name"] for code in indicator_codes]
for col in metric_columns:
    if col not in education_wide.columns:
        education_wide[col] = np.nan

education_wide = education_wide[
    ["Country_Name", "Country_Code", "Region", "Income_Group", "Year"] + metric_columns
].sort_values(["Region", "Country_Name", "Year"]).reset_index(drop=True)

education_wide.to_csv(OUTPUT_DIR / "wdi_education.csv", index=False, encoding="utf-8-sig")

print(f"Rows exported: {len(education_wide):,}")
print(f"Countries exported: {education_wide['Country_Code'].nunique():,}")
print(f"Years exported: {education_wide['Year'].min()}-{education_wide['Year'].max()}")
print(f"Main CSV: {OUTPUT_DIR / 'wdi_education.csv'}")

education_wide.head(10)


Rows exported: 3,537
Countries exported: 192
Years exported: 2000-2023
Main CSV: D:\Lab2_TQH\data_output\wdi_education.csv


,Country_Name,Country_Code,Region,Income_Group,Year,Primary_Net_Enrollment,Secondary_Net_Enrollment,Adult_Literacy_Rate,Education_Expenditure_GDP
0,American Samoa,ASM,East Asia & Pacific,High income,2006,NaN,NaN,NaN,14.71705
1,Australia,AUS,East Asia & Pacific,High income,2000,94.39800,NaN,NaN,NaN
2,Australia,AUS,East Asia & Pacific,High income,2001,93.85651,NaN,NaN,NaN
3,Australia,AUS,East Asia & Pacific,High income,2002,93.85948,NaN,NaN,NaN
4,Australia,AUS,East Asia & Pacific,High income,2003,94.85270,NaN,NaN,NaN
5,Australia,AUS,East Asia & Pacific,High income,2004,95.22550,NaN,NaN,NaN
6,Australia,AUS,East Asia & Pacific,High income,2005,95.50656,NaN,NaN,NaN
7,Australia,AUS,East Asia & Pacific,High income,2006,95.63832,NaN,NaN,NaN
8,Australia,AUS,East Asia & Pacific,High income,2007,96.41764,NaN,NaN,NaN
9,Australia,AUS,East Asia & Pacific,High income,2008,96.53968,NaN,NaN,NaN


In [ ]:
final_quality = pd.DataFrame(
    [
        {
            "Column": col,
            "Non_Null_Cells": int(education_wide[col].notna().sum()),
            "Countries_With_Data": int(education_wide.loc[education_wide[col].notna(), "Country_Code"].nunique()),
            "Min_Value": float(education_wide[col].min(skipna=True)),
            "Max_Value": float(education_wide[col].max(skipna=True)),
        }
        for col in metric_columns
    ]
)

final_quality


,Column,Non_Null_Cells,Countries_With_Data,Min_Value,Max_Value
0,Primary_Net_Enrollment,2144,143,26.827810,99.944810
1,Secondary_Net_Enrollment,1637,133,3.279680,99.911640
2,Adult_Literacy_Rate,294,41,26.830000,100.000000
3,Education_Expenditure_GDP,2610,139,0.000004,15.515512


## Ket luan cho cong viec 2.1 va 2.2

- 4/4 indicators giao duc trong ke hoach deu co trong WDI va duoc ghi nhan trong `wdi_education_indicator_audit.csv`.
- Du lieu da loai aggregate rows, giu metadata `Region` va `Income_Group` de dung filter trong Tableau.
- Missing values duoc xu ly than trong: chi noi suy khoang trong ngan toi da 2 nam, khong extrapolate dau/cuoi chuoi, va loai cap country-indicator co khoang thieu noi bo qua dai.
- File chinh cho Tableau la `data_output/wdi_education.csv`.
